# Max to Python Synth — Before You Start

⏱ About 10 minutes (installation not included)

This notebook does not teach synthesis. It answers three questions: **what this tutorial is for, how hard it is, and how to set up the environment.** At the end is a cell that checks your setup; once it makes sound you are ready for Episode 1.

---

## 1. What this tutorial is for

You can already build synths in Max / RNBO: you know what `phasor~`, `cycle~`, `buffer~`, `adsr~` and `lores~` are and you can patch them.

Over 10 episodes this tutorial translates **the concepts you already understand**, one at a time, into Python. Each episode starts from a Max object you know, builds its Python counterpart, and lets you **hear the result**.

After 10 episodes you will have:

- a complete wavetable synth you wrote yourself (oscillator + filter + two envelopes + modulation), about 100 lines of Python in total
- a parameter and preset system
- the ability to generate "sound ↔ parameter" datasets in bulk
- a way to send wavetables and presets back to Max

### Why a Max user should learn this

Not to replace Max. The two are good at different things:

| | Max / RNBO | Python |
|---|---|---|
| Playing live, turning knobs, building instruments | ✅ excellent | ❌ poor |
| Auto-generating ten thousand sounds with their parameters | 😖 awkward | ✅ one loop |
| Analysing sound, plotting, experimenting | limited | ✅ excellent |
| Machine learning | essentially not possible | ✅ where the mainstream tools live |

If you want to do audio analysis, build datasets, or head toward audio machine learning, Python is unavoidable. And "already understanding synths" is the best starting point for it — you do not have to learn DSP concepts and a programming language at once, only the latter.

### What this tutorial does **not** teach

- real-time audio (the Python synth "computes, then plays"; you cannot play it live)
- graphical interfaces or plugins (VST / AU)
- the whole Python language — only the small part needed for a synth
- mathematical derivations — filters and the FFT are used as ready-made objects

## 2. Difficulty

### What you need to know already

- **Max basics:** the difference between signals (objects with `~`) and messages; some use of `phasor~`, `cycle~`, `buffer~`, `adsr~` and a filter
- **Synth basics:** what an oscillator, a filter, an envelope and an LFO each do
- Experience with `gen~` helps (Episode 6 uses `history` for comparison), but you can follow without it

### What you do **not** need

- **Python: none at all is fine.** Each new piece of syntax is explained on first use and matched to a Max object
- **Maths:** only arithmetic, `sin` and powers. No calculus, no complex numbers

### Difficulty and time per episode

| Ep. | Topic | Time | Difficulty | New Python |
|---|---|---|---|---|
| 1 | sound is an array | 25 min | ★☆☆ | arrays, `[ ]` indexing, `def` functions |
| 2 | phase and basic waveforms | 25 min | ★☆☆ | `%`, `np.where`, first `for` loop |
| 3 | wavetable oscillator | 30 min | ★★☆ | using an array as indices into another |
| 4 | wavetable position | 25 min | ★★☆ | 2-D arrays |
| 5 | envelope | 25 min | ★★☆ | `np.interp`, lists and `append` |
| 6 | filter and the sample loop | 30 min | ★★★ | a `for` loop with "memory" |
| 7 | modulation | 30 min | ★★★ | `np.cumsum`, `2 ** x` |
| 8 | the full synth and presets | 25 min | ★★☆ | `dict` |
| 9 | batch rendering | 25 min | ★★☆ | random numbers, file I/O, f-strings |
| 10 | seeing sound, back to Max | 25 min | ★★☆ | spectrum, spectrogram |

About 4.5 hours in total. **One episode per sitting is recommended.**

Episodes 6 and 7 are the two hardest (the filter loop, and phase accumulation for pitch modulation). Getting stuck there is normal — in Max those ideas are hidden inside the audio engine, and this is the first time you write them out by hand.

### How each episode is laid out

1. **Opening:** which musical element it controls, the matching Max patch, what you will hear
2. **Body:** every step goes "small numbers first → real numbers → listen → look at a plot"
3. **`#` comments in the code:** the tricky lines say what they are doing right beside them
4. **Common mistakes:** a table of error messages and their causes
5. **Exercises:** three, with answers folded underneath (click "Answer" to open)
6. **Recap:** a Max ↔ Python table

## 3. Setting up the environment

Five things are needed:

| | What it does | Minimum version |
|---|---|---|
| **Python** | the language itself | 3.9 |
| **JupyterLab** | the program that opens and runs `.ipynb` notebooks | any recent version |
| **numpy** | math on arrays | 1.20 |
| **scipy** | reading and writing wav files | 1.6 |
| **matplotlib** | plotting | 3.4 |

(These notebooks were tested with Python 3.14, numpy 2.4 and scipy 1.18.)

Pick **one** of the three routes below.

### Route A: Anaconda (recommended, least effort)

Anaconda is one installer that **includes all five** of the above.

1. Download Anaconda Distribution (free) from anaconda.com and install it
2. Open **Anaconda Navigator** and click Launch under **JupyterLab**
3. JupyterLab opens in your browser. Find this tutorial's folder in the file panel on the left and double-click `01_….ipynb`

### Route B: you already have Python — install with pip

Open Terminal (macOS) or Command Prompt (Windows) and run:

```
python3 -m pip install numpy scipy matplotlib jupyterlab
```

When that is done, `cd` into this tutorial's folder and run:

```
jupyter lab
```

### Route C: install nothing — Google Colab

1. Open colab.research.google.com (a Google account is required)
2. File → Upload notebook, and upload an episode's `.ipynb`
3. numpy, scipy and matplotlib are already installed, so it runs straight away

Note: in Colab the files that Episodes 1, 9 and 10 save (wav, json) land in Colab's temporary storage and have to be downloaded by hand from the file panel on the left.

### VS Code works too

With Python and the libraries above installed, add the **Jupyter** extension in VS Code and it opens `.ipynb` files directly.

## 4. Notebook basics

- Each grey box is a **cell**. Click into it and press **Shift + Enter** to run it and move to the next
- **Run from top to bottom, in order.** Later cells use what earlier cells computed
- After changing a number in a cell, **re-run that cell and the cells below it** for the change to take effect. This differs from Max: there a change is live, here you "recompute"
- To start over: menu **Kernel → Restart Kernel and Run All Cells**
- To try something of your own: select a cell and press **B** to add an empty one below
- Text sections (like this one) go into edit mode on a double-click; Shift + Enter restores them

## 5. Checking the environment

Run the cell below.

**You should see:** four version lines, ending with `everything is installed`.

If you get `ModuleNotFoundError: No module named '...'`, that library is missing; go back to section 3.

In [ ]:
import sys
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio

print("Python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("scipy      ", scipy.__version__)
print("matplotlib ", matplotlib.__version__)
print("everything is installed")

### Can it make sound

⚠️ Turn your system volume down a little first.

Run the cell below; a player appears. Press play.

**You should hear:** two short beeps, the second higher than the first.

You do not need to understand this code yet — Episode 1 goes through it line by line.

In [ ]:
sr = 44100
t = np.arange(int(sr * 0.25)) / sr                         # a time table of 0.25 seconds
fade = np.interp(t, [0, 0.01, 0.24, 0.25], [0, 1, 1, 0])   # 10 ms fades at both ends to prevent clicks
beep_low = np.sin(2 * np.pi * 440 * t) * fade
beep_high = np.sin(2 * np.pi * 660 * t) * fade
gap = np.zeros(int(sr * 0.1))                              # 0.1 seconds of silence

Audio(0.3 * np.concatenate([beep_low, gap, beep_high]), rate=sr, normalize=False)

### Can it plot

**You should see:** a plot showing a few cycles of a sine.

In [ ]:
plt.figure(figsize=(9, 2.5))
plt.plot(t[:400], beep_low[:400])
plt.xlabel("time (s)")
plt.title("if you can see this, plotting works")
plt.show()

## 6. Troubleshooting

| Symptom | What to do |
|---|---|
| `ModuleNotFoundError` | A library is missing. Run the pip line from Route B again; with Anaconda, make sure JupyterLab was launched from Anaconda Navigator |
| The player appears but there is no sound | Check whether the browser tab is muted; check the system output device; try Chrome |
| Plots do not appear | Add the line `%matplotlib inline` at the top of that cell and run it again |
| A cell shows `[*]` on the left for a long time | It is still computing. From Episode 6 on some cells take a few seconds. Beyond a minute, use Kernel → Restart |
| `NameError: name '...' is not defined` | An earlier cell was skipped. Kernel → Restart Kernel and Run All Cells |

## 7. Files these notebooks create

All are written to the notebook's own folder and can be deleted at any time:

| Ep. | Files |
|---|---|
| 1 | `ep01_melody.wav` |
| 9 | `dataset/` (20 wavs + `params.json`) |
| 10 | `export/wavetable.wav`, `export/presets.json` |

## 8. Contents

| Ep. | Notebook | Max counterpart |
|---|---|---|
| 0 | Before you start (this one) | — |
| 1 | No `dac~`: sound is an array | `cycle~`, `dac~`, `scope~` |
| 2 | `phasor~`: phase and basic waveforms | `phasor~` |
| 3 | `buffer~` + `wave~`: the wavetable oscillator | `buffer~`, `wave~` |
| 4 | Wavetable position: morphing between waveforms | Serum's WT POS |
| 5 | `adsr~`: the envelope | `adsr~`, `*~` |
| 6 | `lores~`: the filter and the sample loop | `onepole~`, `lores~`, gen~ `history` |
| 7 | Modulation: signals controlling parameters | signals into parameter inlets, LFO |
| 8 | One function is a synth: parameters and presets | RNBO patch, `param` |
| 9 | Batch rendering: what is hard in Max | — |
| 10 | Seeing sound, and back to Max | `spectroscope~`, `buffer~ read`, `dict read` |

Every episode can be opened and run on its own (the functions it needs from earlier episodes are carried along in a "toolbox" cell at the top), but the material builds up in order, so **follow the order the first time through.**

Once the environment check passes, open Episode 1.